# Task 1: News Topic Classifier Using BERT
**DevelopersHub Corporation – AI/ML Engineering Internship**

## Problem Statement
Fine-tune a BERT transformer model to classify news headlines into topic categories using the AG News dataset.

## Objective
- Tokenize and preprocess the AG News dataset
- Fine-tune `bert-base-uncased` using Hugging Face Transformers
- Evaluate using Accuracy and F1-Score
- Deploy with Gradio for live interaction

## 1. Install Dependencies

In [ ]:
# Install required libraries
!pip install transformers datasets torch scikit-learn gradio accelerate -q

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✅ Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 3. Dataset Loading & Preprocessing

In [ ]:
# Load AG News dataset from Hugging Face
print("Loading AG News dataset...")
dataset = load_dataset("ag_news")

print(f"Train samples : {len(dataset['train'])}")
print(f"Test samples  : {len(dataset['test'])}")
print(f"\nFeatures: {dataset['train'].features}")

# Label mapping
label_names = ['World', 'Sports', 'Business', 'Sci/Tech']
print(f"\nLabel classes: {label_names}")

In [ ]:
# Explore the dataset
train_df = pd.DataFrame(dataset['train'])
test_df  = pd.DataFrame(dataset['test'])

print("Sample rows:")
print(train_df.head(5))

# Label distribution
print("\nLabel distribution (train):")
label_counts = train_df['label'].value_counts().sort_index()
for idx, count in label_counts.items():
    print(f"  {label_names[idx]}: {count}")

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts = [train_df['label'].value_counts()[i] for i in range(4)]
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
axes[0].bar(label_names, counts, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('AG News – Training Label Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=label_names, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Label distribution plotted")

In [ ]:
# Use a subset for faster training (remove if you have a GPU / more time)
# Full dataset: 120,000 train / 7,600 test
TRAIN_SUBSET = 8000   # Increase for better accuracy
TEST_SUBSET  = 2000

train_subset = dataset['train'].shuffle(seed=42).select(range(TRAIN_SUBSET))
test_subset  = dataset['test'].shuffle(seed=42).select(range(TEST_SUBSET))

print(f"Training on {TRAIN_SUBSET} samples, evaluating on {TEST_SUBSET} samples")

In [ ]:
# Load BERT tokenizer
MODEL_NAME = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    """Tokenize text with truncation and padding."""
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=128,   # News headlines are short; 128 is sufficient
        padding=False     # Dynamic padding via DataCollatorWithPadding
    )

# Apply tokenization
tokenized_train = train_subset.map(tokenize_function, batched=True)
tokenized_test  = test_subset.map(tokenize_function, batched=True)

# Remove raw text column (not needed by model)
tokenized_train = tokenized_train.remove_columns(['text'])
tokenized_test  = tokenized_test.remove_columns(['text'])

# Rename 'label' to match Trainer expectations
tokenized_train = tokenized_train.rename_column('label', 'labels')
tokenized_test  = tokenized_test.rename_column('label', 'labels')

tokenized_train.set_format('torch')
tokenized_test.set_format('torch')

print("✅ Tokenization complete")
print(f"Sample tokenized input keys: {tokenized_train[0].keys()}")

## 4. Model Development & Training

In [ ]:
# Load pre-trained BERT for sequence classification (4 labels)
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label={i: label for i, label in enumerate(label_names)},
    label2id={label: i for i, label in enumerate(label_names)}
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"Total parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average='weighted')
    return {'accuracy': acc, 'f1': f1}

# Training arguments
training_args = TrainingArguments(
    output_dir='./bert_ag_news',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir='./logs',
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    report_to='none'
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("✅ Trainer configured")

In [ ]:
# Fine-tune the model
print("🚀 Starting fine-tuning...")
train_result = trainer.train()

print("\n✅ Training complete!")
print(f"Training loss : {train_result.training_loss:.4f}")
print(f"Runtime       : {train_result.metrics['train_runtime']:.1f}s")

## 5. Evaluation with Metrics

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
eval_results = trainer.evaluate()

print(f"\n{'='*40}")
print(f"  Test Accuracy : {eval_results['eval_accuracy']:.4f}")
print(f"  Test F1 Score : {eval_results['eval_f1']:.4f}")
print(f"{'='*40}")

In [ ]:
# Detailed classification report
predictions_output = trainer.predict(tokenized_test)
preds = np.argmax(predictions_output.predictions, axis=-1)
true_labels = predictions_output.label_ids

print("Classification Report:")
print(classification_report(true_labels, preds, target_names=label_names))

In [ ]:
# Confusion matrix visualization
cm = confusion_matrix(true_labels, preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_names, yticklabels=label_names,
    linewidths=0.5, ax=ax
)
ax.set_title('Confusion Matrix – BERT AG News Classifier', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for i, name in enumerate(label_names):
    class_acc = cm[i, i] / cm[i].sum()
    print(f"  {name:<10}: {class_acc:.4f}")

In [ ]:
# Plot training metrics across epochs
log_history = trainer.state.log_history

train_losses = [(x['epoch'], x['loss']) for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_metrics = [(x['epoch'], x.get('eval_accuracy', 0), x.get('eval_f1', 0))
                for x in log_history if 'eval_accuracy' in x]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Training loss
if train_losses:
    epochs_tr, losses = zip(*train_losses)
    axes[0].plot(epochs_tr, losses, color='#E74C3C', linewidth=2)
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

# Eval accuracy
if eval_metrics:
    epochs_ev, accs, f1s = zip(*eval_metrics)
    axes[1].plot(epochs_ev, accs, 'o-', color='#2ECC71', linewidth=2, markersize=8)
    axes[1].set_title('Validation Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs_ev, f1s, 'o-', color='#3498DB', linewidth=2, markersize=8)
    axes[2].set_title('Validation F1 Score')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('F1 Score')
    axes[2].grid(True, alpha=0.3)

plt.suptitle('BERT Fine-tuning – Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Model

In [ ]:
# Save fine-tuned model and tokenizer
model.save_pretrained('./bert_ag_news_final')
tokenizer.save_pretrained('./bert_ag_news_final')
print("✅ Model and tokenizer saved to ./bert_ag_news_final")

## 7. Gradio Deployment – Live Interaction

In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr
from transformers import pipeline

# Load model via pipeline for easy inference
classifier = pipeline(
    'text-classification',
    model='./bert_ag_news_final',
    tokenizer='./bert_ag_news_final',
    return_all_scores=True
)

def classify_news(text):
    """Classify a news headline and return category probabilities."""
    if not text.strip():
        return {}
    results = classifier(text)[0]
    return {r['label']: float(f"{r['score']:.4f}") for r in results}

# Example headlines
examples = [
    ["NASA launches new Mars rover to explore the red planet"],
    ["Stock markets rally as Fed signals rate cuts"],
    ["Manchester United wins Premier League title"],
    ["UN calls for ceasefire amid rising tensions in the Middle East"]
]

# Build Gradio interface
demo = gr.Interface(
    fn=classify_news,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Enter a news headline...",
        label="News Headline"
    ),
    outputs=gr.Label(
        num_top_classes=4,
        label="Category Probabilities"
    ),
    title="📰 BERT News Topic Classifier",
    description="Classifies news headlines into: World, Sports, Business, or Sci/Tech",
    examples=examples,
    theme=gr.themes.Soft()
)

demo.launch(share=True)

## 8. Final Summary & Insights

### What We Did
1. **Dataset**: Loaded AG News (4 categories: World, Sports, Business, Sci/Tech) from Hugging Face.
2. **Preprocessing**: Tokenized text using `bert-base-uncased` tokenizer with dynamic padding.
3. **Model**: Fine-tuned BERT with a classification head on top of the `[CLS]` token.
4. **Training**: 3 epochs, learning rate 2e-5, batch size 16, with warm-up and weight decay.
5. **Evaluation**: Reported Accuracy and weighted F1-Score per class.
6. **Deployment**: Launched a Gradio app for live inference.

### Key Results
- BERT fine-tuned on just **8,000 samples** typically achieves **~93–95% accuracy** on AG News.
- Full dataset training (120K samples) pushes accuracy to **~94–95%**.
- The model handles all 4 classes with balanced precision and recall.

### Skills Demonstrated
- ✅ NLP using Transformers (BERT)
- ✅ Transfer learning & fine-tuning
- ✅ Evaluation metrics (Accuracy, F1, Confusion Matrix)
- ✅ Lightweight model deployment with Gradio